Implementazione Fine-tuning di Bert

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW
from torch.utils.data import DataLoader

# Caricamento del modello pre-addestrato e del tokenizer
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Ottimizzatore con learning rate differenziato
optimizer = AdamW([
    {"params": model.bert.parameters(), "lr": 2e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
])

# Loop di addestramento (singola epoca)
model.train()
for batch in dataloader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

    loss = outputs.loss
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

Implementazione concettuale di un layer LoRA

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = alpha / rank

        # Pesi pre-addestrati: congelati
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features), requires_grad=False
        )

        # Componenti LoRA: addestrabili
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        # Inizializzazione (Kaiming uniform)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        # Proiezione base + aggiornamento LoRA
        base = nn.functional.linear(x, self.weight)

        lora_update = nn.functional.linear(
            nn.functional.linear(x, self.lora_A),
            self.lora_B
        ) * self.scaling

        return base + lora_update